# lld-net-pcb-ml &mdash; YOLOv12n Baseline Training

Repository: [**lld-net-pcb-ml**](https://github.com/usarrahim/lld-net-pcb-ml).

Trains the YOLOv12n baseline on the PKU-Market-PCB dataset.
Workflow: Drive &rarr; local `/content` cache &rarr; train &rarr; sync results back to Drive &rarr; evaluate on val/test &rarr; sample inference + ONNX export.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
%cd /content/drive/MyDrive/lld-net-pcb-ml

In [ ]:
!nvidia-smi
!pip -q install --upgrade pip
!pip -q install ultralytics pyyaml

In [ ]:
import os
import shutil
from pathlib import Path
import yaml
import ultralytics
from ultralytics import YOLO

print('Ultralytics:', ultralytics.__version__)

In [ ]:
REPO_ROOT          = Path('/content/drive/MyDrive/lld-net-pcb-ml')
DRIVE_DATASET_DIR  = REPO_ROOT / 'PCB_DATA_YOLO'
DRIVE_DATA_YAML    = DRIVE_DATASET_DIR / 'data.yaml'
DRIVE_RUNS_DIR     = REPO_ROOT / 'runs_pcb'

LOCAL_DATASET_DIR  = Path('/content/PCB_DATA_LOCAL')
LOCAL_DATA_YAML    = LOCAL_DATASET_DIR / 'data.yaml'
LOCAL_PROJECT_DIR  = Path('/content/runs_pcb_local')

if not DRIVE_DATA_YAML.exists():
    raise FileNotFoundError(f'data.yaml not found on Drive: {DRIVE_DATA_YAML}')
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('REPO_ROOT      :', REPO_ROOT)
print('LOCAL_DATASET  :', LOCAL_DATASET_DIR)

In [ ]:
# Local SSD cache: ~5-10x faster epochs than reading the dataset from Drive.
if LOCAL_DATASET_DIR.exists():
    shutil.rmtree(LOCAL_DATASET_DIR)
shutil.copytree(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)

with open(LOCAL_DATA_YAML, 'r', encoding='utf-8') as f:
    yml = yaml.safe_load(f)
yml['path']  = str(LOCAL_DATASET_DIR)
yml['train'] = 'images/train'
yml['val']   = 'images/val'
yml['test']  = 'images/test'
with open(LOCAL_DATA_YAML, 'w', encoding='utf-8') as f:
    yaml.safe_dump(yml, f, sort_keys=False)
print('Local cache:', LOCAL_DATASET_DIR)

In [ ]:
for split in ['train', 'val', 'test']:
    img_dir = LOCAL_DATASET_DIR / 'images' / split
    lbl_dir = LOCAL_DATASET_DIR / 'labels' / split
    n_img = len(list(img_dir.glob('*'))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    print(f'{split:5s} -> images: {n_img:5d}, labels: {n_lbl:5d}')

In [ ]:
MODEL_WEIGHTS = 'yolo12n.pt'   # auto-downloaded by Ultralytics on first use
RUN_NAME      = 'yolo12_pcb_baseline'

IMG_SIZE     = 640
EPOCHS       = 100
BATCH        = 16
DEVICE       = 0
WORKERS      = 4
PATIENCE     = 30
SAVE_PERIOD  = 5
SEED         = 42

In [ ]:
LOCAL_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
run_dir   = LOCAL_PROJECT_DIR / RUN_NAME
last_ckpt = run_dir / 'weights' / 'last.pt'

# If a previous run already lives on Drive, copy it back to local SSD so resume can pick up.
drive_run_dir = DRIVE_RUNS_DIR / RUN_NAME
if drive_run_dir.exists() and not run_dir.exists():
    shutil.copytree(drive_run_dir, run_dir)

if last_ckpt.exists():
    print('Resuming from:', last_ckpt)
    model = YOLO(str(last_ckpt))
    train_results = model.train(resume=True)
else:
    model = YOLO(MODEL_WEIGHTS)
    train_results = model.train(
        data        = str(LOCAL_DATA_YAML),
        epochs      = EPOCHS,
        imgsz       = IMG_SIZE,
        batch       = BATCH,
        device      = DEVICE,
        workers     = WORKERS,
        project     = str(LOCAL_PROJECT_DIR),
        name        = RUN_NAME,
        exist_ok    = True,
        pretrained  = True,
        cache       = True,
        verbose     = True,
        patience    = PATIENCE,
        save_period = SAVE_PERIOD,
        seed        = SEED,
        deterministic = True,
    )

best_pt = run_dir / 'weights' / 'best.pt'
last_pt = run_dir / 'weights' / 'last.pt'
if not best_pt.exists():
    raise FileNotFoundError(f'best.pt not found: {best_pt}')
print('best.pt:', best_pt)

In [ ]:
# Sync the run folder back to Drive so nothing is lost when the Colab session resets.
drive_run_dir = DRIVE_RUNS_DIR / RUN_NAME
if drive_run_dir.exists():
    shutil.rmtree(drive_run_dir)
shutil.copytree(run_dir, drive_run_dir)
print('Synced to:', drive_run_dir)

In [ ]:
best_model = YOLO(str(best_pt))
val_metrics  = best_model.val(data=str(LOCAL_DATA_YAML), split='val',  imgsz=IMG_SIZE, device=DEVICE)
test_metrics = best_model.val(data=str(LOCAL_DATA_YAML), split='test', imgsz=IMG_SIZE, device=DEVICE)
print('VAL  :', val_metrics.results_dict)
print('TEST :', test_metrics.results_dict)

In [ ]:
# Sample predictions on the test split (used later as report figures).
PRED_NAME = f'{RUN_NAME}_predict'
best_model.predict(
    source  = str(LOCAL_DATASET_DIR / 'images' / 'test'),
    imgsz   = IMG_SIZE,
    conf    = 0.25,
    save    = True,
    project = str(LOCAL_PROJECT_DIR),
    name    = PRED_NAME,
    exist_ok= True,
    verbose = False,
)

drive_pred_dir = DRIVE_RUNS_DIR / PRED_NAME
if drive_pred_dir.exists():
    shutil.rmtree(drive_pred_dir)
shutil.copytree(LOCAL_PROJECT_DIR / PRED_NAME, drive_pred_dir)
print('Predictions synced to:', drive_pred_dir)

In [ ]:
# Optional: ONNX export for deployment.
onnx_path = best_model.export(format='onnx', imgsz=IMG_SIZE)
onnx_p = Path(onnx_path)
if onnx_p.exists():
    drive_onnx = DRIVE_RUNS_DIR / RUN_NAME / 'weights' / onnx_p.name
    drive_onnx.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(onnx_p, drive_onnx)
    print('ONNX:', drive_onnx)